# TPI — Detección de Parkinson mediante análisis tiempo-frecuencia de voz
**Señales y Sistemas 2026**

---

| # | Módulo | Contenido |
|---|--------|-----------|
| 0 | Instalación y configuración | dependencias, imports, constantes globales |
| 1 | Preprocesamiento | carga de audio, normalización, segmentación |
| 2 | Descomposición wavelet | DWT db4 multinivel (9 niveles) |
| 3 | Descriptores bioacústicos | jitter, shimmer, energía relativa |
| 4 | Visualizaciones | espectrogramas, escalograma, dispersión PD/HC |
| 5 | Dataset | pipeline completo → `features_pcgita.csv` |
| 5b | Expansión sintética | cópula gaussiana → `features_pcgita_expandido.csv` |
| 6 | Ejecución | corre el pipeline completo |

## Módulo 0 — Instalación y configuración

In [ ]:
!pip install librosa PyWavelets scipy matplotlib seaborn pandas sdv -q

In [ ]:
import numpy as np                                # operaciones matriciales y cálculo numérico
import pywt                                       # PyWavelets: DWT multinivel (wavedec / waverec)
import librosa                                    # carga .wav → array numpy, calcula STFT
import librosa.display                            # specshow: espectrogramas con ejes calibrados
import matplotlib.pyplot as plt                   # gráficos 2D (espectrogramas, escalograma, dispersión)
import seaborn as sns                             # heatmaps para matrices de correlación
import pandas as pd                               # DataFrames y exportación CSV
from pathlib import Path                          # rutas multiplataforma (Windows / Linux)
from scipy.signal import find_peaks, hilbert      # picos locales y señal analítica (envolvente)

print('Importaciones OK')

In [ ]:
SR          = 44100   # Frecuencia de muestreo de PC-GITA [Hz]
NIVELES_DWT = 9       # Niveles de descomposición diádica → 10 arrays (cD1–cD9 + cA9)

# Bandas frecuenciales resultantes con SR = 44100 Hz y NIVELES_DWT = 9
#
#  Índice en coeffs │ Array  │ Banda [Hz]
# ─────────────────┼────────┼──────────────────────────────
#  coeffs[0]       │ cA9    │     0 –    43 Hz
#  coeffs[1]       │ cD9    │    43 –    86 Hz  ← jitter (D9)
#  coeffs[2]       │ cD8    │    86 –   172 Hz  ← jitter (D8)
#  coeffs[3]       │ cD7    │   172 –   344 Hz  ← jitter (D7)
#  coeffs[4]       │ cD6    │   344 –   689 Hz
#  coeffs[5]       │ cD5    │   689 –  1378 Hz
#  coeffs[6]       │ cD4    │  1378 –  2756 Hz  ← shimmer / energía (D4)
#  coeffs[7]       │ cD3    │  2756 –  5512 Hz  ← shimmer / energía (D3)
#  coeffs[8]       │ cD2    │  5512 – 11025 Hz  ← shimmer / energía (D2)
#  coeffs[9]       │ cD1    │ 11025 – 22050 Hz  ← shimmer / energía (D1)
#
# Índice de Dj en la lista:  idx = NIVELES_DWT + 1 - j  (ej: D1 → 9, D9 → 1)
# Jitter    → D7, D8, D9  (43–344 Hz, rango F0 de la voz humana)
# Shimmer   → D1–D4       (1378–22050 Hz, zona de ruido glótico)
# Energía   → D1–D4       (misma zona, relación E_alta / E_total)

print(f'SR={SR} Hz  |  Niveles DWT={NIVELES_DWT} → {NIVELES_DWT + 1} arrays')

## Módulo 1 — Preprocesamiento

**Por qué librosa:** PyWavelets solo trabaja con arrays numpy. Librosa hace el puente: lee el `.wav` y entrega un array normalizado. También recorta silencios con una sola llamada.

- Entrada : archivo `.wav`
- Salida  : `np.ndarray` shape `(N,)`, float64, normalizado en [-1, 1]

In [ ]:
def cargar_señal(path: Path, sr_objetivo: int = SR) -> tuple:
    """Carga .wav con librosa y normaliza en amplitud."""
    y, sr = librosa.load(path, sr=sr_objetivo, mono=True)  # lee el .wav, remuestrea a SR y convierte a mono
    y = y / (np.max(np.abs(y)) + 1e-9)                     # normaliza amplitud a [-1,1]; 1e-9 evita /0
    return y.astype(np.float64), sr                         # float64 requerido por PyWavelets


def recortar_silencios(y: np.ndarray, sr: int, top_db: int = 20) -> np.ndarray:
    """Elimina tramos con energía < top_db dB bajo el máximo.

    top_db=20 (no 30): con 30 dB se perdían tramos de hipofonía —voz débil de baja
    intensidad— que son diagnósticamente relevantes en la disartria hipocinetica del Parkinson.
    """
    intervalos = librosa.effects.split(y, top_db=top_db)    # devuelve pares (inicio, fin) de zonas activas
    if len(intervalos) == 0:                                 # señal completamente silenciosa → no hay qué recortar
        return y                                             # retorna sin modificar para evitar array vacío
    return np.concatenate([y[i:f] for i, f in intervalos])  # concatena solo los fragmentos con voz


def segmentar(y: np.ndarray, sr: int,
              dur_seg: float = 2.0, solapamiento: float = 0.5) -> list:
    """Ventanas de 2 s con 50 % de solapamiento → 88 200 muestras por ventana.

    ADVERTENCIA — data leakage: los segmentos solapados comparten muestras entre sí.
    Siempre hacer el split train/test agrupando por PACIENTE/ARCHIVO, nunca por segmento.
    """
    largo = int(dur_seg * sr)                                          # tamaño de ventana en muestras (88 200 @ 44100 Hz)
    paso  = int(largo * (1.0 - solapamiento))                          # avance entre ventanas (44 100 muestras = 50 %)
    return [y[i : i + largo] for i in range(0, len(y) - largo + 1, paso)]  # lista de segmentos solapados


print('Módulo 1 OK — cargar_señal | recortar_silencios | segmentar')

## Módulo 2 — Descomposición wavelet

**DWT:** red diádica (s = 2^-j, t = k·2^-j). Solo las aproximaciones se descomponen en cada iteración.

Wavelet fija: **Daubechies-4 (db4)**. 9 niveles → 10 arrays (cA9 + cD1–cD9).

In [ ]:
def dwt_multirresolucion(señal: np.ndarray) -> list:
    """
    DWT multinivel db4 por red diádica (s = 2^-j, τ = k·2^-j).
    Wavelet: Daubechies-4 (4 momentos nulos). Niveles: NIVELES_DWT = 9.

    Entrada : señal  → shape (N,)
    Salida  : coeffs → lista de 10 arrays
                coeffs[0] = cA9  (0–43 Hz)
                coeffs[1] = cD9  (43–86 Hz)
                ...
                coeffs[9] = cD1  (11025–22050 Hz)
    """
    return pywt.wavedec(señal, 'db4', level=NIVELES_DWT)  # descomposición DWT db4 en 9 niveles → 10 arrays


print('Módulo 2 OK — dwt_multirresolucion')

## Módulo 3 — Descriptores bioacústicos

Tres descriptores escalares por segmento, derivados de la DWT db4 (9 niveles).

| Descriptor | Banda | Fenómeno capturado |
|---|---|---|
| Jitter relativo | D7–D9 (43–344 Hz) | Variabilidad ciclo a ciclo de F0 |
| Shimmer relativo | D1–D4 (1378–22050 Hz) | Variabilidad de amplitud (ruido glótico) |
| Energía relativa | D1–D4 / total | Proporción de energía en altas frecuencias |

In [ ]:
def _idx(j: int) -> int:
    """Convierte número de nivel Dj al índice en la lista de pywt.wavedec.
    pywt invierte el orden: D1 (alta frec) queda al final.
    Con NIVELES_DWT=9: D1→9, D4→6, D7→3, D9→1."""
    return NIVELES_DWT + 1 - j   # fórmula de mapeo: D1→índice 9, D9→índice 1


def reconstruir_banda(coeffs: list, niveles: list) -> np.ndarray:
    """Reconstruye la señal filtrando solo los niveles de detalle indicados.
    Anula el resto de coeficientes y aplica pywt.waverec (filtro paso-banda wavelet).

    niveles: lista de enteros, ej. [7, 8, 9] para jitter, [1, 2, 3, 4] para shimmer.
    """
    coeffs_filt = [np.zeros_like(c) for c in coeffs]   # inicializa todos los coeficientes en cero
    for j in niveles:
        coeffs_filt[_idx(j)] = coeffs[_idx(j)].copy()  # copia solo los niveles deseados al nuevo array
    return pywt.waverec(coeffs_filt, 'db4')             # reconstruye la señal vía síntesis wavelet inversa


def calcular_jitter(coeffs: list, sr: int = SR) -> float:
    """
    Jitter relativo: variabilidad ciclo a ciclo de la frecuencia fundamental.
    Opera sobre D7–D9 (43–344 Hz), banda de F0 pura tras filtrado wavelet.

    Retorna: jitter relativo (adimensional); 0 si hay menos de 3 picos.
    """
    banda    = reconstruir_banda(coeffs, [7, 8, 9])             # señal filtrada en banda de F0 (43–344 Hz)
    dist_min = max(2, int(sr / 344))                            # distancia mínima entre picos: 1 ciclo a 344 Hz ≈ 128 muestras

    peaks, _ = find_peaks(banda, distance=dist_min, height=0)   # detecta máximos locales con altura positiva
    if len(peaks) < 3:                                          # menos de 3 picos → imposible calcular jitter
        return 0.0

    periodos = np.diff(peaks).astype(np.float64)                # distancias en muestras entre picos consecutivos
    return float(np.mean(np.abs(np.diff(periodos))) / (np.mean(periodos) + 1e-9))  # jitter = variación de periodos / periodo medio


def calcular_shimmer(coeffs: list, sr: int = SR) -> float:
    """
    Shimmer relativo: variabilidad de amplitud en la banda de alta frecuencia.
    Opera sobre D1–D4 (1378–22050 Hz), zona de ruido glótico y soplosidad.

    Retorna: shimmer relativo (adimensional); 0 si no hay suficientes ventanas.
    """
    banda      = reconstruir_banda(coeffs, [1, 2, 3, 4])        # señal filtrada en zona de ruido glótico (D1–D4)
    envolvente = np.abs(hilbert(banda))                         # envolvente instantánea mediante señal analítica (Hilbert)

    ventana    = max(1, int(sr * 0.01))                         # 441 muestras = 10 ms ≈ duración de un ciclo glótico
    n_ventanas = len(envolvente) // ventana                     # número total de ventanas sin solapamiento
    if n_ventanas < 3:                                          # menos de 3 ventanas → shimmer indefinido
        return 0.0

    amplitudes = np.array([
        np.max(envolvente[i * ventana : (i + 1) * ventana])     # amplitud máxima en cada ventana de 10 ms
        for i in range(n_ventanas)
    ])
    return float(np.mean(np.abs(np.diff(amplitudes))) / (np.mean(amplitudes) + 1e-9))  # shimmer = variación de amplitudes / amplitud media


def calcular_energia_espectral(coeffs: list) -> float:
    """
    Energía relativa en D1–D4 respecto a la energía total (Parseval wavelet).
    E_rel = Σ(D1²+D2²+D3²+D4²) / Σ(todos los coeficientes²)

    Valores altos → mayor proporción de energía en altas frecuencias
    (indicativo de ruido glótico / soplosidad parkinsónica).
    """
    energia_total = sum(np.sum(c ** 2) for c in coeffs)             # Parseval: suma de cuadrados de todos los coeficientes
    if energia_total < 1e-12:                                        # señal prácticamente silenciosa → evita división por cero
        return 0.0
    energia_alta = sum(np.sum(coeffs[_idx(j)] ** 2) for j in [1, 2, 3, 4])  # energía acumulada en D1–D4 (altas frecuencias)
    return float(energia_alta / energia_total)                       # cociente: fracción de energía en zona de ruido glótico


print('Módulo 3 OK — calcular_jitter | calcular_shimmer | calcular_energia_espectral')

## Modulo 4 — Visualizaciones

Cuatro gráficos comparativos PD vs HC:

1. Espectrograma de **banda ancha** — alta resolución temporal (pulsos glóticos)
2. Espectrograma de **banda angosta** — alta resolución frecuencial (armónicos)
3. **Escalograma** DWT — mapa tiempo × nivel D1–D9
4. **Dispersión** jitter vs shimmer — un punto por sujeto

In [ ]:
# ── Tema visual unificado ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'      : 110,          # resolución de pantalla en puntos por pulgada
    'figure.facecolor': '#0f1117',    # fondo oscuro del área exterior de la figura
    'axes.facecolor'  : '#1a1d2e',    # fondo oscuro del área de graficado
    'text.color'      : 'white',      # color de todos los textos
    'axes.labelcolor' : 'white',      # color de las etiquetas de ejes
    'xtick.color'     : 'white',      # color de las marcas del eje X
    'ytick.color'     : 'white',      # color de las marcas del eje Y
})


def graficar_espectrograma_ancho(señal_pd, señal_hc, sr=SR):
    """
    Espectrograma de banda ancha: ventana corta (~5 ms), alta resolución temporal.
    Permite ver pulsos glóticos individuales y fluctuaciones de amplitud (shimmer).
    """
    n_fft = 256        # ventana FFT de 256 muestras ≈ 5.8 ms @ 44100 Hz → alta resolución temporal
    hop   = n_fft // 4 # avance entre frames = 64 muestras (75 % solapamiento)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))                       # dos paneles lado a lado
    for ax, señal, titulo in zip(axes, [señal_pd, señal_hc], ['PD', 'HC']):
        S    = librosa.stft(señal.astype(np.float32), n_fft=n_fft, hop_length=hop)  # STFT: matriz compleja frecuencia × tiempo
        S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)                       # convierte magnitud a escala logarítmica (dB)
        librosa.display.specshow(S_db, sr=sr, hop_length=hop,
                                 x_axis='time', y_axis='hz', ax=ax, cmap='magma')   # dibuja espectrograma con ejes calibrados
        ax.set_title(f'Espectrograma banda ancha — {titulo}')
        ax.set_ylim(0, 5000)                                                         # limita a 5 kHz para foco en voz

    plt.suptitle('Banda ancha: alta resolución temporal (pulsos glóticos)', color='white')
    plt.tight_layout()
    plt.savefig('espectrograma_banda_ancha.png', bbox_inches='tight')   # exporta a PNG sin bordes blancos
    plt.show()


def graficar_espectrograma_angosto(señal_pd, señal_hc, sr=SR):
    """
    Espectrograma de banda angosta: ventana larga (~46 ms), alta resolución frecuencial.
    Permite separar armónicos individuales y visualizar ondulaciones de F0 (jitter).
    """
    n_fft = 2048       # ventana FFT de 2048 muestras ≈ 46 ms @ 44100 Hz → alta resolución frecuencial
    hop   = n_fft // 8 # avance entre frames = 256 muestras (87.5 % solapamiento)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, señal, titulo in zip(axes, [señal_pd, señal_hc], ['PD', 'HC']):
        S    = librosa.stft(señal.astype(np.float32), n_fft=n_fft, hop_length=hop)  # STFT con ventana larga
        S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)                       # magnitud a dB
        librosa.display.specshow(S_db, sr=sr, hop_length=hop,
                                 x_axis='time', y_axis='hz', ax=ax, cmap='magma')
        ax.set_title(f'Espectrograma banda angosta — {titulo}')
        ax.set_ylim(0, 2000)                                                         # limita a 2 kHz para ver armónicos de F0

    plt.suptitle('Banda angosta: alta resolución frecuencial (armónicos)', color='white')
    plt.tight_layout()
    plt.savefig('espectrograma_banda_angosta.png', bbox_inches='tight')
    plt.show()


def graficar_escalograma(coeffs_pd, coeffs_hc):
    """
    Escalograma DWT: mapa 2D tiempo × nivel D1–D9, magnitud como intensidad cromática.
    Muestra la distribución de energía multirresolución de la db4.
    """
    fig, axes   = plt.subplots(1, 2, figsize=(14, 5))
    etiquetas_y = [f'D{j}' for j in range(1, NIVELES_DWT + 1)]   # etiquetas D1–D9 para el eje Y

    for ax, coeffs, titulo in zip(axes, [coeffs_pd, coeffs_hc], ['PD', 'HC']):
        longitud_ref = len(coeffs[_idx(5)])   # D5 como longitud de referencia para la interpolación

        # Cada nivel Dj tiene distinto número de coeficientes → se interpola a una longitud común
        matriz = np.array([
            np.interp(
                np.linspace(0, 1, longitud_ref),                        # eje destino: longitud_ref puntos uniformes
                np.linspace(0, 1, len(np.abs(coeffs[_idx(j)]))),        # eje origen: longitud original de Dj
                np.abs(coeffs[_idx(j)])                                 # valores a interpolar (magnitud de coeficientes)
            )
            for j in range(1, NIVELES_DWT + 1)                         # itera D1→D9
        ])

        im = ax.imshow(matriz, aspect='auto', origin='lower',
                       cmap='magma', interpolation='nearest')           # dibuja la matriz como imagen de calor
        ax.set_yticks(range(NIVELES_DWT))                               # una marca por nivel DWT
        ax.set_yticklabels(etiquetas_y)                                 # etiqueta D1–D9 en el eje Y
        ax.set_xlabel('Tiempo (coeficientes interpolados)')
        ax.set_ylabel('Nivel DWT')
        ax.set_title(f'Escalograma db4 — {titulo}')
        plt.colorbar(im, ax=ax, label='|coef.|')                        # barra de colores con magnitud

    plt.suptitle('Escalograma wavelet: densidad de energía multirresolución', color='white')
    plt.tight_layout()
    plt.savefig('escalograma_wavelet.png', bbox_inches='tight')
    plt.show()


def graficar_dispersion_jitter_shimmer(
        df: pd.DataFrame,
        titulo: str   = 'Distribución jitter-shimmer: PD vs HC',
        filename: str = 'dispersion_jitter_shimmer.png'):
    """
    Dispersión jitter vs shimmer, un punto por sujeto, color por grupo.
    Entrada: DataFrame con columnas 'jitter_mean', 'shimmer_mean', 'grupo'
             (grupo=1 PD, grupo=0 HC).
    """
    colores   = {0: '#4fc3f7', 1: '#ef5350'}            # azul claro = HC, rojo = PD
    etiquetas = {0: 'HC (sano)', 1: 'PD (Parkinson)'}   # texto para la leyenda

    n_pd = (df['grupo'] == 1).sum()   # cuenta sujetos PD para mostrar en leyenda
    n_hc = (df['grupo'] == 0).sum()   # cuenta sujetos HC para mostrar en leyenda

    fig, ax = plt.subplots(figsize=(8, 6))
    for grupo, sub in df.groupby('grupo'):                        # itera sobre grupo 0 (HC) y grupo 1 (PD)
        ax.scatter(sub['jitter_mean'], sub['shimmer_mean'],
                   c=colores[grupo], label=f"{etiquetas[grupo]} (n={len(sub)})",
                   alpha=0.75, s=60, edgecolors='white', linewidths=0.4)  # puntos semitransparentes con borde blanco

    ax.set_xlabel('Jitter relativo (D7–D9)')
    ax.set_ylabel('Shimmer relativo (D1–D4)')
    ax.set_title(titulo)
    ax.legend()
    plt.tight_layout()
    plt.savefig(filename, bbox_inches='tight')
    plt.show()


print('Módulo 4 OK — graficar_espectrograma_ancho | graficar_espectrograma_angosto | graficar_escalograma | graficar_dispersion_jitter_shimmer')

## Módulo 5 — Construcción del dataset

Procesa la vocal `/a/` de PC-GITA y exporta `features_pcgita.csv` (100 filas, una por sujeto).

```
PC-GITA_per_task_44100Hz/modulated vowels/
├── pd/
│   └── A/   ← 50 archivos PD  (etiqueta 1)
└── hc/
    └── A/   ← 50 archivos HC  (etiqueta 0)
```

In [ ]:
DATA_DIR  = Path(r'C:\Users\mateo\OneDrive\Desktop\UNI Sheiße\S&S\PC-GITA_per_task_44100Hz\PC-GITA_per_task_44100Hz\modulated vowels')
LABEL_MAP = {'pd': 1, 'hc': 0}   # PD=1 (Parkinson Disease), HC=0 (Healthy Control)
VOCAL     = 'A'                   # subcarpeta con los .wav de la vocal /a/ sostenida

print(f'DATA_DIR : {DATA_DIR}')
print(f'PD existe: {(DATA_DIR / "pd" / VOCAL).exists()}')
print(f'HC existe: {(DATA_DIR / "hc" / VOCAL).exists()}')

In [ ]:
def construir_dataset(data_dir: Path = DATA_DIR,
                      vocal: str = VOCAL) -> pd.DataFrame:
    """
    Itera sobre pd/A/ y hc/A/, procesa cada .wav y agrega los descriptores
    bioacústicos a nivel de sujeto (promedio y desviación estándar sobre los segmentos).

    Salida: DataFrame con columnas:
        sujeto_id, grupo, jitter_mean, jitter_std,
        shimmer_mean, shimmer_std, energia_mean, energia_std
    """
    registros = []   # acumula un dict por sujeto procesado

    for clase, etiqueta in LABEL_MAP.items():           # itera 'pd'→1 y 'hc'→0
        carpeta = data_dir / clase / vocal              # construye ruta: …/pd/A/ o …/hc/A/
        if not carpeta.exists():                        # carpeta ausente → avisa y salta
            print(f"[AVISO] Carpeta no encontrada: {carpeta}  — omitiendo.")
            continue

        # "*_a.wav": doble seguridad contra mezcla de tareas si la estructura del dataset cambia.
        # PC-GITA nombra los archivos de vocal /a/ como AVPEPUDEA0001_a.wav (sufijo _a).
        archivos = sorted(carpeta.glob(f"*_{vocal.lower()}.wav"))   # lista ordenada de .wav de la vocal
        print(f"  {clase}/{vocal}: {len(archivos)} archivos")

        for archivo in archivos:
            try:
                señal, sr = cargar_señal(archivo)           # carga y normaliza el .wav
                señal     = recortar_silencios(señal, sr)   # elimina silencios iniciales/finales
                segmentos = segmentar(señal, sr)            # divide en ventanas de 2 s con 50 % solapamiento

                jitters, shimmers, energias = [], [], []   # listas para acumular descriptores por segmento
                for seg in segmentos:
                    coeffs = dwt_multirresolucion(seg)              # descomposición DWT db4 del segmento
                    jitters.append(calcular_jitter(coeffs, sr))     # jitter del segmento → lista
                    shimmers.append(calcular_shimmer(coeffs, sr))   # shimmer del segmento → lista
                    energias.append(calcular_energia_espectral(coeffs))  # energía relativa → lista

                registros.append({
                    'sujeto_id'    : archivo.stem,          # nombre de archivo sin extensión como ID
                    'grupo'        : etiqueta,              # 1=PD, 0=HC
                    'jitter_mean'  : np.mean(jitters),      # promedio de jitter sobre todos los segmentos
                    'jitter_std'   : np.std(jitters),       # desviación estándar de jitter
                    'shimmer_mean' : np.mean(shimmers),     # promedio de shimmer
                    'shimmer_std'  : np.std(shimmers),      # desviación estándar de shimmer
                    'energia_mean' : np.mean(energias),     # promedio de energía relativa
                    'energia_std'  : np.std(energias),      # desviación estándar de energía relativa
                })

            except Exception as e:
                print(f"  [ERROR] {archivo.name}: {e}")     # registra el error sin detener el pipeline

    if not registros:
        raise RuntimeError(
            "No se encontraron datos.\n"
            f"  Verificar DATA_DIR / LABEL_MAP / VOCAL.\n"
            f"  Ruta esperada: {data_dir / 'pd' / vocal}  y  {data_dir / 'hc' / vocal}"
        )

    df = pd.DataFrame(registros)                            # convierte la lista de dicts a DataFrame
    print(f"\nDataset construido: {len(df)} sujetos — "
          f"{(df['grupo'] == 1).sum()} PD / {(df['grupo'] == 0).sum()} HC")
    return df


def exportar_csv(df: pd.DataFrame, ruta: str = "features_pcgita.csv") -> None:
    """Exporta el DataFrame a CSV para Orange (grupo=clase, sujeto_id para LOSO)."""
    df.to_csv(ruta, index=False)   # guarda sin columna de índice numérico de pandas
    print(f"CSV exportado: {ruta}  ({len(df)} filas × {len(df.columns)} columnas)")


print('Módulo 5 OK — construir_dataset | exportar_csv')

## Módulo 5b — Expansión sintética del dataset

Genera datos sintéticos a partir del CSV real (100 sujetos) usando **Cópulas Gaussianas** (SDV).
La cópula aprende las distribuciones marginales y la matriz de correlación entre features,
luego muestrea nuevas observaciones coherentes con esa estructura.

- Entrada : `features_pcgita.csv` — 100 filas × 8 columnas
- `sujeto_id` se excluye antes de sintetizar (es un identificador, no una feature)
- Salida   : `features_pcgita_expandido.csv` — ~600 filas (100 reales + 500 sintéticas)

In [ ]:
from sdv.metadata import SingleTableMetadata          # inferencia automática del esquema de columnas
from sdv.single_table import GaussianCopulaSynthesizer # sintetizador basado en cópulas gaussianas
from sdv.evaluation.single_table import evaluate_quality  # métricas de fidelidad estadística

# ── 1. Cargar el dataset real ─────────────────────────────────────────────────
data      = pd.read_csv('features_pcgita.csv')         # carga las 100 filas reales (50 PD + 50 HC)
data_feat = data.drop(columns=['sujeto_id'])            # elimina sujeto_id: es un ID, no una feature predictiva

# ── 2. Ajustar el sintetizador ────────────────────────────────────────────────
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=data_feat)          # infiere tipos de columna (numérico, categórico, etc.)

synthesizer = GaussianCopulaSynthesizer(metadata)
# Aprende una distribución marginal por columna y las une con una cópula gaussiana,
# preservando la estructura de correlación del dataset real.
synthesizer.fit(data_feat)                              # ajusta el modelo sobre los 100 sujetos reales

# ── 3. Generar datos sintéticos ───────────────────────────────────────────────
datos_sinteticos = synthesizer.sample(num_rows=500)     # muestrea 500 filas nuevas de la distribución aprendida
print(f"Distribución grupo en datos sintéticos: {datos_sinteticos['grupo'].round().value_counts().to_dict()}")

# ── 4. Limpiar etiquetas y asignar IDs sintéticos ────────────────────────────
datos_sinteticos['grupo']     = datos_sinteticos['grupo'].round().astype(int).clip(0, 1)  # redondea y fuerza 0 o 1
datos_sinteticos['sujeto_id'] = [f'SYNTH_{i:04d}' for i in range(len(datos_sinteticos))]
# Prefijo SYNTH_ permite distinguir filas sintéticas de reales en análisis posteriores

# ── 5. Combinar y exportar ────────────────────────────────────────────────────
dataset_expandido = pd.concat([data, datos_sinteticos[data.columns]], ignore_index=True)  # apila reales + sintéticos
dataset_expandido.to_csv('features_pcgita_expandido.csv', index=False)                    # exporta sin índice de pandas
print(f"CSV exportado: features_pcgita_expandido.csv  ({len(dataset_expandido)} filas × {len(dataset_expandido.columns)} columnas)")

# ── 6. Validación: calidad de los datos sintéticos ───────────────────────────
quality_report = evaluate_quality(
    real_data=data_feat,
    synthetic_data=datos_sinteticos.drop(columns=['sujeto_id']),  # excluye ID sintético para comparar solo features
    metadata=metadata
)
# Métricas: Column Shapes (KS test por columna) + Column Pair Trends (correlaciones)

# ── 7. Comparación visual de matrices de correlación ─────────────────────────
feat_cols        = ['jitter_mean', 'jitter_std', 'shimmer_mean', 'shimmer_std', 'energia_mean', 'energia_std']
matriz_real      = data[feat_cols].corr()                  # matriz de correlación de Pearson sobre datos reales
matriz_sintetica = datos_sinteticos[feat_cols].corr()      # misma matriz sobre datos sintéticos

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(matriz_real,      annot=True, fmt='.2f', cmap='coolwarm', ax=axes[0]).set_title('Correlaciones — Real (100 sujetos)')
sns.heatmap(matriz_sintetica, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1]).set_title('Correlaciones — Sintético (500 muestras)')
plt.suptitle('Validación: preservación de correlaciones entre features', color='white')
plt.tight_layout()
plt.savefig('validacion_correlaciones.png', bbox_inches='tight')   # exporta comparación a PNG
plt.show()

print('Módulo 5b OK — datos sintéticos generados y validados')

## Módulo 6 — Ejecución

Corre el pipeline completo: construye el dataset real, genera los gráficos comparativos PD/HC
y exporta el dataset expandido con datos sintéticos.

> Requiere haber ejecutado todos los módulos de definición anteriores (0–5b).

In [ ]:
# ── 1. Construir y exportar el dataset real ───────────────────────────────────
df = construir_dataset()    # procesa todos los .wav y agrega descriptores por sujeto
exportar_csv(df)            # guarda features_pcgita.csv (100 filas)

# ── 2. Cargar señales de ejemplo para las visualizaciones ────────────────────
patron_vocal = f'*_{VOCAL.lower()}.wav'                             # patrón glob para archivos de vocal /a/
archivos_pd  = sorted((DATA_DIR / 'pd' / VOCAL).glob(patron_vocal))  # lista ordenada de archivos PD
archivos_hc  = sorted((DATA_DIR / 'hc' / VOCAL).glob(patron_vocal))  # lista ordenada de archivos HC

señal_pd, _ = cargar_señal(archivos_pd[0])       # carga el primer sujeto PD como representante
señal_pd    = recortar_silencios(señal_pd, SR)    # elimina silencios del sujeto PD

señal_hc, _ = cargar_señal(archivos_hc[0])       # carga el primer sujeto HC como representante
señal_hc    = recortar_silencios(señal_hc, SR)    # elimina silencios del sujeto HC

seg_pd    = segmentar(señal_pd, SR)[0]            # primer segmento de 2 s del sujeto PD (88 200 muestras)
seg_hc    = segmentar(señal_hc, SR)[0]            # primer segmento de 2 s del sujeto HC
coeffs_pd = dwt_multirresolucion(seg_pd)          # DWT db4 sobre el segmento PD → 10 arrays
coeffs_hc = dwt_multirresolucion(seg_hc)          # DWT db4 sobre el segmento HC → 10 arrays

print(f'Sujeto PD de ejemplo : {archivos_pd[0].stem}')
print(f'Sujeto HC de ejemplo : {archivos_hc[0].stem}')

# ── 3. Gráficos comparativos PD / HC ─────────────────────────────────────────
graficar_espectrograma_ancho(seg_pd, seg_hc)      # espectrograma banda ancha (resolución temporal)
graficar_espectrograma_angosto(seg_pd, seg_hc)    # espectrograma banda angosta (resolución frecuencial)
graficar_escalograma(coeffs_pd, coeffs_hc)        # escalograma DWT tiempo × nivel D1–D9

# Muestra original (100 sujetos reales)
graficar_dispersion_jitter_shimmer(
    df,
    titulo   = 'Distribución jitter-shimmer: PD vs HC — muestra original',
    filename = 'dispersion_jitter_shimmer.png'
)

# Muestra ampliada (100 reales + 500 sintéticos = 600 sujetos)
df_expandido = pd.read_csv('features_pcgita_expandido.csv')   # carga el dataset expandido con datos sintéticos
graficar_dispersion_jitter_shimmer(
    df_expandido,
    titulo   = 'Distribución jitter-shimmer: PD vs HC — muestra ampliada (cópula gaussiana)',
    filename = 'dispersion_jitter_shimmer_expandido.png'
)